# 🧠 Reasoning Pruning: Interactive Exploration & Live Overthinking Discovery

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/avrymi-asraf/reasoning-pruning-agy/blob/master/notebooks/01_explore_pruning.ipynb)

This notebook provides an **end-to-end, live interactive laboratory** for generating reasoning traces with Generator model $G$, auditing them live for skippable overthinking spans using Decision model $D$, visually rendering trace diffs, and extracting $(x \to y)$ transition datasets for QLoRA fine-tuning.

### 🎯 The Core Pipeline:
$$\text{Question } q \xrightarrow[\text{generate\_trace}]{\text{Generator } G} \text{Trace } (s_1..s_n) \xrightarrow[\text{find\_first\_skip}]{\text{Decision } D} \text{Skip } s_k \xrightarrow[\text{extract\_transition}]{\text{Pair}} (x \to y) \xrightarrow[\text{rollout\_pruning}]{\text{Recursive Rollout}} \text{PT Dataset}$$

| Live Tool | Purpose | Output Structure |
|---|---|---|
| `rp.generate_trace` | Prompts generator $G$ and segments reasoning steps | `ReasoningTrace` |
| `rp.find_first_skip` | Audits trace live with decision model $D$ | `PruneDecision` |
| `rp.render_trace_diff` | Renders color-coded HTML diff of pruned steps | `HTML` / `RichPanel` |
| `rp.extract_transition` | Isolates $(x \to y)$ next-step training pair | `TransitionExample` |
| `rp.rollout_pruning` | Multi-depth iterative pruning and continuation | `RolloutResult` |
| `rp.build_pt_dataset` | Assembles Hugging Face Dataset across questions | `datasets.Dataset` |

## 1. Setup, Environment & API Keys

Detects Google Colab to automatically clone the repo and install dependencies in editable mode (`pip install -e .`). Configure your LiteLLM model endpoints and API keys below.

In [ ]:
# 1. Environment bootstrap (Google Colab vs local workspace)
import os
import sys

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
if IN_COLAB:
    print("🚀 Running on Google Colab. Setting up environment and repository...")
    REPO_DIR = "/content/reasoning-pruning-agy"
    if not os.path.exists(REPO_DIR):
        !git clone https://github.com/avrymi-asraf/reasoning-pruning-agy.git {REPO_DIR}
    %cd {REPO_DIR}
    !pip install -q -e .
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)
    print("✅ reasoning-pruning repository cloned and installed!")
else:
    print("💻 Running in local workspace.")

import json
import re
import pandas as pd
from IPython.display import HTML, display

# Import core reasoning_pruning tools
import reasoning_pruning as rp
from reasoning_pruning.types import (
    ReasoningTrace,
    PruneDecision,
    TransitionExample,
    RolloutResult,
)

# Configure API Keys for LiteLLM providers:
# os.environ["GEMINI_API_KEY"] = "your-gemini-key"
# os.environ["OPENAI_API_KEY"] = "your-openai-key"
# os.environ["ANTHROPIC_API_KEY"] = "your-anthropic-key"
# os.environ["DEEPSEEK_API_KEY"] = "your-deepseek-key"
# os.environ["HF_TOKEN"] = "your-hf-token"

print(f"✅ reasoning_pruning v{rp.__version__} loaded successfully.")

## 2. Model Selection & Reasoning Benchmark Spectrum

Choose the Generator model $G$ and Decision Auditor $D$. Any LiteLLM-supported model (Gemini, OpenAI, Anthropic, DeepSeek, Ollama, HuggingFace) can be specified.

In [ ]:
# Configure Generator G and Decision Auditor D
# Examples: "gemini/gemini-2.5-flash", "gpt-4o-mini", "claude-3-5-haiku-20241022", "deepseek/deepseek-chat"
MODEL_G = os.environ.get("RP_MODEL_G", "gemini/gemini-2.5-flash" if "GEMINI_API_KEY" in os.environ else "gpt-4o-mini")
MODEL_D = os.environ.get("RP_MODEL_D", "gemini/gemini-2.5-flash" if "GEMINI_API_KEY" in os.environ else "gpt-4o-mini")

print(f"Generator Model (G): {MODEL_G}")
print(f"Decision Model  (D): {MODEL_D}")

# Define 6-Family Cognitive Reasoning Spectrum
TASK_SPECTRUM = [
    # 1. Arithmetic & Word Math
    {
        "task_id": "math_01",
        "category": "Arithmetic & Word Math",
        "question": "Janet buys 3 packs of 12 eggs. She bakes 2 cakes using 4 eggs each. How many eggs does she have left?",
        "ground_truth": "28",
    },
    {
        "task_id": "math_02",
        "category": "Arithmetic & Word Math",
        "question": "A baker has 45 cookies. He packages 5 cookies per box. How many boxes can he fill completely?",
        "ground_truth": "9",
    },
    # 2. Commonsense & Physical Logic
    {
        "task_id": "common_01",
        "category": "Commonsense & Physical Logic",
        "question": "Can a penguin fly over the English Channel?",
        "ground_truth": "No",
    },
    {
        "task_id": "common_02",
        "category": "Commonsense & Physical Logic",
        "question": "If you put a hot cup of coffee in a freezer at -18°C, will it become hotter or colder?",
        "ground_truth": "Colder",
    },
    # 3. Multi-hop & Deductive Logic
    {
        "task_id": "multihop_01",
        "category": "Multi-hop & Deductive Logic",
        "question": "Christopher Nolan directed Inception. Which country was Christopher Nolan born in?",
        "ground_truth": "United Kingdom",
    },
    {
        "task_id": "multihop_02",
        "category": "Multi-hop & Deductive Logic",
        "question": "All roses are flowers. All flowers need sunlight to survive. Does a rose need sunlight to survive?",
        "ground_truth": "Yes",
    },
    # 4. Algorithmic & State Tracking
    {
        "task_id": "algo_01",
        "category": "Algorithmic & State Tracking",
        "question": "A coin starts heads up. You flip it once, then leave it alone, then flip it again. Is the coin currently heads or tails?",
        "ground_truth": "Heads",
    },
    # 5. Extractive & Span QA
    {
        "task_id": "extract_01",
        "category": "Extractive & Span QA",
        "question": "Context: The Apollo 11 mission landed on the Moon on July 20, 1969, carrying Neil Armstrong and Buzz Aldrin. Question: In what year did Apollo 11 land on the Moon?",
        "ground_truth": "1969",
    },
    # 6. Axiomatic & Math Properties
    {
        "task_id": "axiom_01",
        "category": "Axiomatic & Math Properties",
        "question": "Is 144 an even number?",
        "ground_truth": "Yes",
    },
]

display(pd.DataFrame(TASK_SPECTRUM))

## 3. Live Trace Generation (`rp.generate_trace`)

Calls Generator $G$ live via `rp.generate_trace` to produce a full reasoning trajectory, which is automatically segmented into discrete step boundaries.

In [ ]:
# Select problem from spectrum
sample_problem = TASK_SPECTRUM[2]  # "Can a penguin fly over the English Channel?"
question_text = sample_problem["question"]

print(f"🎯 Question: {question_text}")
print(f"Generating live reasoning trace with Model G ({MODEL_G})...")

# Live generation call
live_trace = rp.generate_trace(
    question=question_text,
    model=MODEL_G,
    temperature=0.7,
)

print(f"\n✅ Generated {len(live_trace.steps)} reasoning steps ({live_trace.token_count} tokens):")
for i, step in enumerate(live_trace.steps):
    print(f"  [{i}] {step}")

## 4. Live Overthinking Audit (`rp.find_first_skip`)

Passes the live `ReasoningTrace` to Decision Auditor model $D$. The auditor determines whether intermediate steps contain conversational preambles, question restatements, redundant verification loops, or irrelevant detours that can be safely skipped.

In [ ]:
print(f"Auditing trace live with Decision Model ({MODEL_D})...")

# Live decision call
live_decision = rp.find_first_skip(
    trace=live_trace,
    decision_model=MODEL_D,
    temperature=0.0,
)

print("\n📋 Prune Decision Result:")
print(f"  Can Skip:        {live_decision.can_skip}")
if live_decision.can_skip:
    print(f"  Skip Span:       Steps [{live_decision.skip_start_idx} .. {live_decision.skip_end_idx}]")
    print(f"  Skipped Steps:   {live_decision.skipped_steps}")
    print(f"  Next Kept Step:  {live_decision.next_step}")
    print(f"  Auditor Reason:  {live_decision.reason}")
else:
    print(f"  Reason:          {live_decision.reason}")

## 5. Live Visual Trace Diff Rendering (`rp.render_trace_diff`)

Renders a transparent, color-coded HTML diff:
- 🟩 **Green text**: Kept context prefix ($x$)
- 🟥 **Red strikethrough**: Removable/redundant thoughts ($s_k$)
- 🟦 **Cyan text**: Next useful deduction target ($y$)

In [ ]:
# Render interactive HTML diff from live objects
html_diff = rp.render_trace_diff(live_trace, live_decision, as_html=True)
display(HTML(html_diff))

## 6. Live Transition Extraction (`rp.extract_transition`)

Transforms the audited trace into a training transition example $(x \to y)$ where the model learns to bypass the redundant span directly.

In [ ]:
if live_decision.can_skip:
    live_transition = rp.extract_transition(
        trace=live_trace,
        decision=live_decision,
        depth=1,
        example_id=f"{sample_problem[task_id]}_d1",
    )
    
    print("=" * 60)
    print(f"🎯 EXTRACTED PRUNING-TRANSITION PAIR: {live_transition.id}")
    print("=" * 60)
    print(f"📌 Input Context (x):\n{live_transition.input_x}\n")
    print(f"🚀 Target Continuation (y):\n{live_transition.target_y}\n")
    print(f"✂️ Skipped Thoughts:\n{live_transition.skipped_steps}\n")
    print(f"💡 Audit Justification:\n{live_transition.skip_reason}")
else:
    print("No skippable steps found for this trace; transition extraction not applicable.")

## 7. Live Multi-Depth Recursive Rollout (`rp.rollout_pruning`)

Executes recursive multi-depth pruning across iterations on a complex problem. At each depth $d$, the pruned prefix is fed back into $G$, auditing the continuation for secondary redundancies until the final answer is reached.

In [ ]:
# Run multi-depth rollout on an arithmetic word problem
rollout_q = TASK_SPECTRUM[0]["question"]
print(f"🔬 Running Live Multi-Depth Rollout for: {rollout_q}")

rollout_res = rp.rollout_pruning(
    question=rollout_q,
    model_g=MODEL_G,
    model_d=MODEL_D,
    max_depth=3,
)

print(f"\n✅ Rollout complete! Total Depths: {rollout_res.total_depths}, Transitions Extracted: {len(rollout_res.transitions)}")
print(f"Original Tokens: {rollout_res.original_tokens} ➔ Final Tokens: {rollout_res.final_tokens} ({rollout_res.compression_pct:.1f}% compression)")

for i, tr in enumerate(rollout_res.transitions, 1):
    print(f"\n--- [Transition {i} | Depth {tr.depth}] ---")
    print(f"Input (x):  {tr.input_x[:80]}...")
    print(f"Target (y): {tr.target_y}")
    print(f"Skipped:    {tr.skipped_steps}")
    print(f"Reason:     {tr.skip_reason}")

## 8. Live Pruning-Transition Dataset Builder (`rp.build_pt_dataset`)

Converts a list of questions or benchmark dataset into a Hugging Face `Dataset` ready for 4-bit QLoRA SFT training.

In [ ]:
# Select subset of questions from spectrum
sample_questions = [t["question"] for t in TASK_SPECTRUM[:4]]

print(f"Building Hugging Face PT Dataset live across {len(sample_questions)} benchmark questions...")
hf_dataset = rp.build_pt_dataset(
    questions=sample_questions,
    generator_model=MODEL_G,
    decision_model=MODEL_D,
    max_depth=2,
    max_workers=2,
)

print(f"\n✅ Built Live Dataset with {len(hf_dataset)} transition examples!")
if len(hf_dataset) > 0:
    df_ds = hf_dataset.to_pandas()
    display(df_ds[["id", "depth", "input_x", "target_y", "skip_reason"]].head())

# To synchronize to Hugging Face Hub:
# rp.push_dataset_to_hf(hf_dataset, "your-hf-username/rp-transitions-v1")